In [ ]:
# Hapus dulu yang bikin error
!apt-get remove google-chrome-stable chromium-browser chromium-chromedriver -y > /dev/null
!rm -rf /content/chrome-linux64 /content/chromedriver-linux64

# 1. Download Installer Resmi Google Chrome (.deb)
!wget -q https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb

# 2. Instal pake apt (Supaya dia otomatis instal library yang kurang)
!apt install ./google-chrome-stable_current_amd64.deb -y > /dev/null

# 3. Instal Selenium & Manager Driver
!pip install selenium webdriver-manager beautifulsoup4 pandas > /dev/null

print("✅ Sip! Chrome Resmi so ta-instal. Lanjut ke bawah!")

E: Unable to locate package google-chrome-stable


✅ Sip! Chrome Resmi so ta-instal. Lanjut ke bawah!


# **FB**

In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from bs4 import BeautifulSoup
import time
import pandas as pd

# Setup Chrome supaya ringan (Headless)
options = Options()
options.add_argument('--headless')
options.add_argument('--no-sandbox')
options.add_argument('--disable-dev-shm-usage')
options.add_argument('--disable-gpu')

try:
    print("🚀 Mencoba nyalakan Chrome Resmi...")

    # Pake WebDriver Manager biar dia yang atur driver
    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

    print("🎉 ALAMAKJANG! AKHIRNYA JADI, YER!")

    # --- TES BUKA GRUP MANADO ---
    url = "https://mbasic.facebook.com/groups/280819332106487"
    print(f"Lagi buka: {url}")
    driver.get(url)

    all_texts = []

    # Coba ambil 2 halaman
    for i in range(2):
        print(f"Panen halaman {i+1}...")
        soup = BeautifulSoup(driver.page_source, 'html.parser')

        # Ambil teks di dalam div atau p
        for post in soup.find_all(['div', 'p']):
            teks = post.get_text()
            # Filter: minimal 15 huruf biar bukan sampah
            if len(teks) > 15:
                all_texts.append(teks)

        try:
            # Cari tombol 'Lihat Postingan Lainnya'
            next_link = driver.find_element("partial link text", "Lihat Postingan")
            next_link.click()
            time.sleep(3) # Jeda sopan
        except:
            print("So nda dapa tombol Next.")
            break

    print(f"\n✅ Total dapa {len(all_texts)} baris data!")
    print("Contoh data:", all_texts[:3])

    driver.quit()

except Exception as e:
    print(f"Masih error kote: {e}")

🚀 Mencoba nyalakan Chrome Resmi...
🎉 ALAMAKJANG! AKHIRNYA JADI, YER!
Lagi buka: https://mbasic.facebook.com/groups/280819332106487
Panen halaman 1...
So nda dapa tombol Next.

✅ Total dapa 68 baris data!
Contoh data: ['Log into FacebookEmail or mobile numberPasswordLog inForgot password?Create new account English (US)EspañolFrançais (France)中文(简体)العربيةPortuguês (Brasil)ItalianoMore languages…Sign UpLog InMessengerFacebook LiteVideoMeta PayMeta StoreMeta QuestRay-Ban MetaMeta AIMeta AI more contentInstagramThreadsVoting Information CenterPrivacy PolicyConsumer Health PrivacyPrivacy CenterAboutCreate adCreate PageDevelopersCareersCookiesAd choices TermsHelpContact Uploading & Non-UsersMeta © 2026', 'Log into FacebookEmail or mobile numberPasswordLog inForgot password?Create new account English (US)EspañolFrançais (France)中文(简体)العربيةPortuguês (Brasil)ItalianoMore languages…Sign UpLog InMessengerFacebook LiteVideoMeta PayMeta StoreMeta QuestRay-Ban MetaMeta AIMeta AI more contentInstag

In [ ]:
# --- 1. BERSIH-BERSIH & SETUP ---
!pkill -9 -f chrome
!pkill -9 -f chromedriver

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
import pandas as pd
import getpass
import time

# Opsi Chrome Anti-Deteksi
options = Options()
options.add_argument('--headless=new')
options.add_argument('--no-sandbox')
options.add_argument('--disable-dev-shm-usage')
options.add_argument('--disable-gpu')
options.add_argument('--disable-notifications')
# Trik supaya nda dianggap bot otomatis
options.add_argument('--disable-blink-features=AutomationControlled')

try:
    print("🔄 Menghidupkan Mesin Chrome...")
    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
    print("✅ Mesin Siap! Mari Login.")

    # --- 2. INPUT DATA (Biar aman nda hardcode di script) ---
    email = input("Email FB: ")
    print("Password FB (Ketik, nda muncul huruf, lalu Enter):")
    password = getpass.getpass()

    # --- 3. PROSES LOGIN ---
    print("\n🚀 Meluncur ke Halaman Login...")
    driver.get("https://mbasic.facebook.com/login")

    # Isi Form
    WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.NAME, "email"))).send_keys(email)
    driver.find_element(By.NAME, "pass").send_keys(password)
    driver.find_element(By.NAME, "login").click()

    print("⏳ Sedang memproses login... (Tunggu 5 detik)")
    time.sleep(5)

    # --- 4. HANDLING 'SIMPAN BROWSER' (Ini yg sering bikin gagal) ---
    # Kadang FB tanya "Simpan Browser ini?", torang harus klik OK dulu biar sesinya valid
    if "save-device" in driver.current_url or "checkpoint" in driver.current_url:
        print("⚠️ Muncul halaman verifikasi/simpan browser...")
        try:
            # Cari tombol 'OK' atau 'Simpan'
            tombol_ok = driver.find_element(By.XPATH, "//input[@value='OK'] | //a[contains(text(), 'OK')]")
            tombol_ok.click()
            print("✅ Tombol OK diklik. Sesi diamankan.")
            time.sleep(3)
        except:
            print("   (Tombol OK nda dapa, coba lanjut paksa...)")

    # --- 5. NAVIGASI KE GRUP ---
    # Cek dulu apa betul so masuk
    if "Log In" not in driver.title:
        print("🎉 LOGIN SUKSES! OTW Grup Portal Mongondow/Manado...")

        target_url = "https://mbasic.facebook.com/groups/280819332106487"
        driver.get(target_url)
        time.sleep(5) # Tunggu loading grup

        # --- 6. SCRAPING LOOP (PANEN DATA) ---
        print("\n🚜 MEMULAI PANEN DATA...")
        all_data = []

        # Ambil 5 halaman
        for i in range(5):
            print(f"   📄 Halaman {i+1}...", end=" ")
            soup = BeautifulSoup(driver.page_source, 'html.parser')

            # Cari elemen teks (div, p, span)
            jumlah_dapa = 0
            for el in soup.find_all(['div', 'p', 'span']):
                teks = el.get_text().strip()

                # FILTER KUAT:
                # 1. Minimal 20 huruf
                # 2. Bukan menu FB (Beranda, Profil, dll)
                # 3. Bukan pesan login
                forbidden_words = ["Log In", "Buat Akun", "Facebook", "Lihat Postingan", "Komentar", "Suka", "Bagikan"]
                if len(teks) > 20 and not any(k in teks for k in forbidden_words):
                    if teks not in all_data:
                        all_data.append(teks)
                        jumlah_dapa += 1

            print(f"Dapa {jumlah_dapa} baris baru.")

            # Klik Next Page
            try:
                next_btn = driver.find_element(By.PARTIAL_LINK_TEXT, "Lihat Postingan")
                next_btn.click()
                time.sleep(4)
            except:
                print("   🛑 Tombol Next habis/nda dapa.")
                break

        # --- 7. SIMPAN HASIL ---
        print(f"\n🎉 SELESAI! Total data bersih: {len(all_data)}")

        if len(all_data) > 0:
            print("--- CONTOH DATA ASLI ---")
            for d in all_data[:3]:
                print(f"👉 {d}")

            df = pd.DataFrame(all_data, columns=['Teks_Manado_FB'])
            df.to_csv('dataset_fb_manado_final.csv', index=False)
            print("\n💾 File 'dataset_fb_manado_final.csv' so tersimpan aman!")
        else:
            print("⚠️ Data kosong. Cek screenshot 'debug_gagal.png'.")
            driver.save_screenshot('debug_gagal.png')

    else:
        print("\n❌ GAGAL LOGIN. Password salah ato akun kena checkpoint.")
        print(f"Posisi terakhir: {driver.current_url}")
        driver.save_screenshot('gagal_login.png')

except Exception as e:
    print(f"\n❌ ERROR FATAL: {e}")

finally:
    # Matikan driver biar RAM lega
    if 'driver' in locals():
        driver.quit()
        print("🔌 Driver dimatikan.")

INFO:WDM:====== WebDriver manager ======
INFO:WDM:Get LATEST chromedriver version for google-chrome
INFO:WDM:Get LATEST chromedriver version for google-chrome
INFO:WDM:Driver [/root/.wdm/drivers/chromedriver/linux64/144.0.7559.109/chromedriver-linux64/chromedriver] found in cache


🔄 Menghidupkan Mesin Chrome...
✅ Mesin Siap! Mari Login.
Email FB: yermiaturangan07@gmail.com
Password FB (Ketik, nda muncul huruf, lalu Enter):
··········

🚀 Meluncur ke Halaman Login...
⏳ Sedang memproses login... (Tunggu 5 detik)
🎉 LOGIN SUKSES! OTW Grup Portal Mongondow/Manado...

🚜 MEMULAI PANEN DATA...
   📄 Halaman 1... Dapa 2 baris baru.
   🛑 Tombol Next habis/nda dapa.

🎉 SELESAI! Total data bersih: 2
--- CONTOH DATA ASLI ---
👉 NoticeYou must log in to continue.
👉 You must log in to continue.

💾 File 'dataset_fb_manado_final.csv' so tersimpan aman!
🔌 Driver dimatikan.


# **YOUTUBE**

In [ ]:
# 1. Instal alatnya dulu (Cuma 3 detik)
!pip install youtube-comment-downloader pandas

# 2. Langsung Panen!
from youtube_comment_downloader import *
import pandas as pd
import itertools

downloader = YoutubeCommentDownloader()

# Ganti URL video lagu Manado/Vlog di sini
# Contoh: Video Lagu Manado Galau yang viral
url = "https://www.youtube.com/watch?v=1FXV5msaKWs"

print(f"🚀 Lagi sedot data dari: {url}")

# Ambil komentar (pake generator biar hemat memori)
comments = downloader.get_comments_from_url(url, sort_by=SORT_BY_POPULAR)

# Kita ambil 100 komentar pertama jo dulu
data_panen = []
limit = 100

print("Sedang memproses...")
for comment in itertools.islice(comments, limit):
    teks = comment['text']
    # Filter sadiki: Hapus yang terlalu pendek
    if len(teks) > 10:
        data_panen.append(teks)

# Simpan hasil
if data_panen:
    df = pd.DataFrame(data_panen, columns=['Komentar_Manado'])
    df.to_csv('dataset_youtube_jadi.csv', index=False)

    print(f"\n🎉 ALAMAKJANG! Sukses dapa {len(data_panen)} komentar!")
    print("Contoh data asli:")
    for d in data_panen[:5]:
        print(f"➡️ {d}")
else:
    print("Yah, kosong. Coba ganti link video lain.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 315.5/315.5 kB 7.8 MB/s eta 0:00:00
🚀 Lagi sedot data dari: https://www.youtube.com/watch?v=1FXV5msaKWs
Sedang memproses...

🎉 ALAMAKJANG! Sukses dapa 90 komentar!
Contoh data asli:
➡️ Sapa manado
Like dibawah ini 
👇
#nomaksa
➡️ terima kasih telah memperacayakan music kepada saya 🙏🔥
➡️ Gw udah nonton YouTube rewind 2019 dari setiap daerah di indo 






YouTube rewind Indonesianya mana?
➡️ Asekkkk... tinggal depe transisi yg kurang dapat deng depe jalan cerita. Kek ada yg ta potong2.. mar so termasuk mantap.. berkembang terus youtuber manado.
➡️ Dpe konsep sebenarnya so bagus. Kalo cuma mo ambe dpe tujuan for "rewind" so dapa.
Cuma qta pe kurang di alur yg berantakan dg transisi yg sama berantakan sih.
Sebenarnya so cukup meyakinkan di awal.
Butuh banyak bljr sama.
Tetap smngttt


# **TWITTER**

In [ ]:
# --- 1. INSTALASI (Kalo belum) ---
!pip install ntscraper pandas

# --- 2. SETUP & KEYWORDS ---
from ntscraper import Nitter
import pandas as pd
import time
import random

scraper = Nitter()

# Kita pake banya kata kunci biar dapa data yang variatif
# Ini kata-kata yang pasti orang Manado pake
keywords = [
    "ngana", "torang", "kita pe", "manado",
    "baku dapa", "kiapa", "so boleh", "jang bagitu"
]

all_tweets = []
target_per_keyword = 50 # Target dapa 50 tweet per kata kunci

print(f"🚀 Memulai Misi Panen Tweet Manado...")
print(f"Target: {len(keywords) * target_per_keyword} data mentah")

# --- 3. EKSEKUSI LOOPING ---
for kata in keywords:
    print(f"\n🔎 Lagi cari keyword: '{kata}'...")

    try:
        # Kita coba tembak server Nitter spesifik biar lebe stabil
        # Kalo gagal, hapus bagian instance=... biar dia cari random
        tweets = scraper.get_tweets(
            kata,
            mode='term',
            number=target_per_keyword,
            instance='https://nitter.privacydev.net'
        )

        dapat = 0
        if 'tweets' in tweets and len(tweets['tweets']) > 0:
            for t in tweets['tweets']:
                teks = t['text']
                # Filter: Ambil yang panjang minimal 15 huruf & bukan link doang
                if len(teks) > 15 and "http" not in teks:
                    all_tweets.append([kata, teks]) # Simpan kategori & teks
                    dapat += 1
            print(f"   ✅ Dapa {dapat} tweet.")
        else:
            print("   ⚠️ Kosong. Server lagi pelit.")

    except Exception as e:
        print(f"   ❌ Error di keyword ini: {e}")

    # Istirahat bentar biar nda dikira bot jahat
    time.sleep(random.uniform(2, 5))

# --- 4. SIMPAN HASIL ---
print(f"\n🎉 SELESAI! Total terkumpul: {len(all_tweets)} tweet.")

if len(all_tweets) > 0:
    # Bikin DataFrame
    df = pd.DataFrame(all_tweets, columns=['Keyword', 'Teks_Manado'])

    # Bersihkan duplikat (kadang ada tweet sama di keyword beda)
    df = df.drop_duplicates(subset=['Teks_Manado'])

    # Simpan
    nama_file = 'dataset_twitter_manado_full.csv'
    df.to_csv(nama_file, index=False)

    print(f"📂 File so tersimpan: {nama_file}")
    print("\n--- Contoh Data ---")
    print(df.head())
else:
    print("Yah, zonk. Coba ganti 'instance' di kode atau coba lagi nanti.")

Testing instances: 100%|██████████| 8/8 [00:02<00:00,  3.25it/s]


🚀 Memulai Misi Panen Tweet Manado...
Target: 400 data mentah

🔎 Lagi cari keyword: 'ngana'...
   ❌ Error di keyword ini: Cannot choose from an empty sequence

🔎 Lagi cari keyword: 'torang'...
   ❌ Error di keyword ini: Cannot choose from an empty sequence

🔎 Lagi cari keyword: 'kita pe'...
   ❌ Error di keyword ini: Cannot choose from an empty sequence

🔎 Lagi cari keyword: 'manado'...
   ❌ Error di keyword ini: Cannot choose from an empty sequence

🔎 Lagi cari keyword: 'baku dapa'...
   ❌ Error di keyword ini: Cannot choose from an empty sequence

🔎 Lagi cari keyword: 'kiapa'...
   ❌ Error di keyword ini: Cannot choose from an empty sequence

🔎 Lagi cari keyword: 'so boleh'...
   ❌ Error di keyword ini: Cannot choose from an empty sequence

🔎 Lagi cari keyword: 'jang bagitu'...
   ❌ Error di keyword ini: Cannot choose from an empty sequence

🎉 SELESAI! Total terkumpul: 0 tweet.
Yah, zonk. Coba ganti 'instance' di kode atau coba lagi nanti.


In [ ]:
!pip install googlesearch-python pandas

from googlesearch import search
import pandas as pd
import time

# Query sakti: Cari kalimat Manado spesifik di dalam situs Twitter
query = 'site:twitter.com "ngana" OR "torang" OR "baku dapa" Manado'

print(f"🚀 Lagi tanya Mbah Google tentang tweet Manado...")

links = []
# Ambil 50 link teratas
for j in search(query, num_results=50, lang="id"):
    links.append(j)
    time.sleep(1) # Sopan sadiki

print(f"Dapa {len(links)} link Twitter yang relevan.")
# Masalahnya: Ini cuma dapa Link, ngna musti klik manual atau pake Selenium buat buka satu-satu.
# TAPI, Google kadang kase "Snippet" (potongan teks) di hasil pencarian.

🚀 Lagi tanya Mbah Google tentang tweet Manado...
Dapa 0 link Twitter yang relevan.


# **IG**

In [ ]:
# 1. Instal library
!pip install instaloader pandas

# 2. Eksekusi
import instaloader
import pandas as pd

# Bikin instance (Tanpa Login biar aman)
L = instaloader.Instaloader()

# Target: Cari akun publik Manado yang aktif (Username jangan salah!)
target_akun = "kotamanado" # Ganti deng akun lain misal: manadopost, dll
print(f"🚀 Lagi intip akun: {target_akun}")

data_ig = []

try:
    posts = instaloader.Profile.from_username(L.context, target_akun).get_posts()

    # Ambil caption dari 20 postingan terakhir
    count = 0
    for post in posts:
        if post.caption:
            data_ig.append(post.caption)
            count += 1

        if count >= 20: # Jangan rakus, nanti IP di-blokir
            break

    # Simpan
    if data_ig:
        df = pd.DataFrame(data_ig, columns=['Caption_IG'])
        df.to_csv('dataset_ig_manado.csv', index=False)
        print(f"\n🎉 Aman! Dapa {len(data_ig)} caption.")
        print("Contoh:", data_ig[:2])
    else:
        print("Kosong kote. Mungkin akun di-private.")

except Exception as e:
    print(f"Aduh error: {e}")
    print("Tips: IG sering blokir IP Colab. Kalo error terus, brarti IP Colab lagi ditandai.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.3/68.3 kB 5.1 MB/s eta 0:00:00
🚀 Lagi intip akun: kotamanado


JSON Query to graphql/query: Expecting value: line 1 column 1 (char 0) [retrying; skip with ^C]



🎉 Aman! Dapa 20 caption.
Contoh: ['Daripada menyesal karena sadar belakangan, mari mulai saling membantu kebersihan kantor', 'Kalo ngoni?\n#fyp #foryou #lari #pagi']


# SKALIAN

In [ ]:
import pandas as pd
import re
import numpy as np
from sklearn.model_selection import train_test_split

# --- 1. FUNGSI MEMBERSIHKAN TEKS (CLEANING) ---
def bersihkan_teks(teks):
    if not isinstance(teks, str): return ""

    # Ganti huruf kecil semua (Case Folding) - Bagus buat BERT awal
    teks = teks.lower()

    # Hapus URL (http/https/www)
    teks = re.sub(r'http\S+|www\S+|https\S+', '', teks, flags=re.MULTILINE)

    # Hapus Mention (@username) dan Hashtag (#)
    teks = re.sub(r'@\w+|#\w+', '', teks)

    # Hapus Emoji dan Simbol Aneh (Simpan titik/koma/tanda tanya/seru)
    teks = re.sub(r'[^a-z0-9\s.,!?]', '', teks)

    # Hapus spasi berlebih
    teks = re.sub(r'\s+', ' ', teks).strip()

    return teks

print("🚀 MEMULAI PROSES PENGGABUNGAN DATA...")

# --- 2. LOAD DATASET (Pastikan nama file sesuai yg tadi torang bikin) ---
data_gabungan = []

# A. Data Facebook
try:
    df_fb = pd.read_csv('dataset_ig_manado.csv')
    # Sesuaikan nama kolom dg yg di CSV FB
    col_name = 'Teks_Manado_FB' if 'Teks_Manado_FB' in df_fb.columns else df_fb.columns[0]
    print(f"✅ Load FB: {len(df_fb)} baris")
    data_gabungan.extend(df_fb[col_name].tolist())
except Exception as e:
    print(f"⚠️ Data IG belum ada/error: {e}")

# B. Data YouTube
try:
    df_yt = pd.read_csv('dataset_youtube_jadi.csv')
    col_name = 'Komentar_Manado' if 'Komentar_Manado' in df_yt.columns else df_yt.columns[0]
    print(f"✅ Load YouTube: {len(df_yt)} baris")
    data_gabungan.extend(df_yt[col_name].tolist())
except Exception as e:
    print(f"⚠️ Data YouTube belum ada/error: {e}")

# C. Data Sintetis (Pabrik Kalimat)
try:
    df_syn = pd.read_csv('dataset_manado_sintetis_1000.csv')
    col_name = 'Teks_Manado' if 'Teks_Manado' in df_syn.columns else df_syn.columns[0]
    print(f"✅ Load Sintetis: {len(df_syn)} baris")
    data_gabungan.extend(df_syn[col_name].tolist())
except Exception as e:
    print(f"⚠️ Data Sintetis belum ada/error: {e}")

# --- 3. PROSES CLEANING & FILTERING ---
print("\n🧹 Sedang membersihkan data...")

df_final = pd.DataFrame(data_gabungan, columns=['teks_raw'])

# Terapkan fungsi cleaning
df_final['teks_bersih'] = df_final['teks_raw'].apply(bersihkan_teks)

# Hapus data kosong atau terlalu pendek (kurang dari 3 huruf)
df_final = df_final[df_final['teks_bersih'].str.len() > 3]

# HAPUS DUPLIKAT (Penting biar training efisien)
jumlah_awal = len(df_final)
df_final = df_final.drop_duplicates(subset=['teks_bersih'])
jumlah_akhir = len(df_final)

print(f"   Dibuang {jumlah_awal - jumlah_akhir} data duplikat/sampah.")
print(f"🎉 TOTAL DATASET BERSIH: {jumlah_akhir} Kalimat.")

# --- 4. SPLITTING (TRAIN vs VALIDATION) ---
# Bagi 85% Train, 15% Test
train, val = train_test_split(df_final['teks_bersih'], test_size=0.15, random_state=42)

print(f"\n📊 Statistik:")
print(f"   - Data Training: {len(train)} baris")
print(f"   - Data Validasi: {len(val)} baris")

# --- 5. SIMPAN KE TXT (Format Dataset BERT) ---
# Kita simpan per baris (line-by-line)
train.to_csv('data_train_manado.txt', index=False, header=False)
val.to_csv('data_val_manado.txt', index=False, header=False)

print("\n💾 SUKSES! File 'data_train_manado.txt' dan 'data_val_manado.txt' siap dipake!")
print("👉 Contoh 5 Kalimat Bersih:")
print(train.head().values)